# Telematics Insurance RTA Project

Notebook prowadzi przez demonstracyjny projekt usage-based insurance / pay-how-you-drive. Laczy generowanie zdarzen telematycznych, Kafka, Spark Structured Streaming, agregacje okienkowe, batchowe trenowanie GLM i okresowe przeliczanie skladki technicznej.

Najwazniejsza zasada biznesowa: pojedynczy event nie zmienia skladki. Eventy sa przetwarzane do flag ryzyka, potem do agregatow, a dopiero zagregowane cechy trafiaja do modelu i scoringu.

## 1. Architektura

Przeplyw danych:

```text
producer_telematics.py -> Kafka telematics_raw
Spark Structured Streaming -> telematics_alerts + data/historical_features
train_glm.py -> models/glm_model.pkl + data/model_outputs
score_premiums.py -> data/premium_history/premium_history.csv + opcjonalnie Kafka premium_updates
```

Streaming odpowiada za biezace przetwarzanie i cechy. Batch odpowiada za trenowanie modelu. Scoring stosuje wytrenowany model do najnowszych agregatow.

## 2. Kafka topics

W terminalu JupyterLab utworz tematy. Broker w srodowisku kursowym powinien byc dostepny jako `broker:9092`.

In [ ]:
!bash ../scripts/create_topics.sh

Podglad topicow i wiadomosci mozna wykonac z terminala:

```bash
kafka-topics.sh --list --bootstrap-server broker:9092
kafka-console-consumer.sh --bootstrap-server broker:9092 --topic telematics_raw --from-beginning --max-messages 5
kafka-console-consumer.sh --bootstrap-server broker:9092 --topic telematics_alerts --from-beginning --max-messages 5
```

## 3. Producent danych telematycznych

Producent generuje 50 kierowcow z ukrytym profilem: `safe`, `average`, `aggressive`, `night_driver`, `urban_driver`. Profil wplywa na predkosc, hamowanie, przyspieszenie, jazde noca i uzycie telefonu. Dane sa wysylane jako JSON do `telematics_raw`.

In [ ]:
# Uruchom w osobnym terminalu, aby notebook nie blokowal kolejnych komorek:
# cd ..
# python producer_telematics.py --drivers 50 --events-per-second 10 --duration-seconds 300 --seed 2026

## 4. Spark Structured Streaming

Skrypt `spark_streaming_features.py` czyta Kafka topic, parsuje JSON przez jawny schema, konwertuje timestamp ISO z koncowka `Z`, dodaje flagi ryzyka, wysyla alerty i zapisuje agregaty parquet.

Wybierz connector zgodny z wersja Sparka i Scali w srodowisku.

In [ ]:
# Spark 3.5 / Scala 2.12:
# !cd .. && spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 spark_streaming_features.py

# Spark 4.0 preview / Scala 2.13:
# !cd .. && spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2 spark_streaming_features.py

### Reguly ryzyka

Przykladowe flagi:

- `is_speeding = speed_kmh > speed_limit_kmh + 10`
- `is_hard_braking = braking_ms2 < -3.0`
- `is_harsh_acceleration = acceleration_ms2 > 2.5`
- `is_sharp_cornering = cornering_g > 0.35`
- `is_night_risk`, `is_bad_weather`, `is_phone_usage`

Alert trafia do `telematics_alerts` tylko dla eventow ryzykownych i zawiera aktywne reguly oraz `risk_score_event`.

### Okna, watermark i agregaty

Skrypt liczy:

- tumbling window 1 minuta,
- sliding window 5 minut / krok 1 minuta,
- watermark 30 sekund.

Dla kazdego kierowcy i okna powstaja m.in. `event_count`, `km_driven`, `avg_speed`, `speeding_ratio`, `risk_events_per_100km`, `exposure_km`.

## 5. Batch: trenowanie GLM

Po zebraniu danych historycznych uruchom trening. Jesli parquet ze streamingu jeszcze nie istnieje, skrypt automatycznie wygeneruje demonstracyjne agregaty syntetyczne, dzieki czemu dalsza czesc notebooka nadal dziala.

Model Poisson GLM uzywa offsetu `log(exposure_km)`, czyli modeluje czestosc szkod wzgledem przejechanych kilometrow.

In [ ]:
import os
os.chdir('..')
!python train_glm.py

In [ ]:
import pandas as pd

coef = pd.read_csv('data/model_outputs/glm_coefficients.csv')
metrics = pd.read_json('data/model_outputs/glm_metrics.json', typ='series')
display(coef)
display(metrics)

Interpretacja znakow wspolczynnikow: dodatni wspolczynnik zwieksza oczekiwana czestosc szkody przy stalej ekspozycji. `exp_coef` pokazuje mnoznik intensywnosci dla jednostkowej zmiany cechy. Przy nadmiernej dyspersji Poisson jest zbyt prosty i nalezy rozwazyc Negative Binomial albo Tweedie.

## 6. Scoring i skladka techniczna

Scoring liczy przewidywana czestosc szkod, risk multiplier i skladke techniczna. Aktualizacja skladki ma limit +/-10%, aby pojedyncza aktualizacja nie powodowala skokow.

In [ ]:
!python score_premiums.py

In [ ]:
premium = pd.read_csv('data/premium_history/premium_history.csv')
display(premium.sort_values('technical_premium', ascending=False).head(10))

## 7. Wizualizacje

Ponizsze wykresy korzystaja z plikow wygenerowanych przez trening i scoring. Przy prawdziwym strumieniu mozna je rozszerzyc o parquet z `data/historical_features/tumbling_1m`.

In [ ]:
import matplotlib.pyplot as plt

features = pd.read_csv('data/model_outputs/training_dataset.csv')
pred = pd.read_csv('data/model_outputs/glm_test_predictions.csv')
premium = pd.read_csv('data/premium_history/premium_history.csv')

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# 1. Liczba eventow ryzyka w czasie - przyblizenie z syntetycznych agregatow
if 'window_id' in features.columns:
    risk_by_window = features.assign(risk_proxy=features['speeding_ratio'] + features['night_event_ratio']).groupby('window_id')['risk_proxy'].sum()
    risk_by_window.plot(figsize=(10, 4), title='Risk proxy w czasie')
    plt.xlabel('Okno')
    plt.ylabel('Risk proxy')
    plt.show()

In [ ]:
# 2. Przekroczenia predkosci wedlug kierowcy
speeding = features.groupby('driver_id')['speeding_ratio'].mean().sort_values(ascending=False).head(15)
speeding.plot(kind='bar', figsize=(10, 4), title='Sredni speeding_ratio wedlug kierowcy')
plt.ylabel('speeding_ratio')
plt.show()

In [ ]:
# 3. Risk score wedlug kierowcy
features['risk_score_driver'] = (
    features['speeding_ratio']
    + features['hard_braking_count_per_100km'] / 100
    + features['harsh_acceleration_count_per_100km'] / 100
    + features['night_event_ratio']
    + features['bad_weather_event_ratio']
)
risk_score = features.groupby('driver_id')['risk_score_driver'].mean().sort_values(ascending=False).head(15)
risk_score.plot(kind='bar', figsize=(10, 4), title='Risk score wedlug kierowcy')
plt.ylabel('risk_score_driver')
plt.show()

In [ ]:
# 4. Predicted claim frequency wedlug kierowcy
freq = pred.groupby('driver_id')['predicted_frequency'].mean().sort_values(ascending=False).head(15)
freq.plot(kind='bar', figsize=(10, 4), title='Predicted claim frequency')
plt.ylabel('predicted_frequency')
plt.show()

In [ ]:
# 5. Techniczna skladka wedlug kierowcy
latest_premium = premium.sort_values('scored_at').groupby('driver_id').tail(1)
latest_premium.set_index('driver_id')['technical_premium'].sort_values(ascending=False).head(15).plot(
    kind='bar', figsize=(10, 4), title='Techniczna skladka wedlug kierowcy'
)
plt.ylabel('PLN')
plt.show()

In [ ]:
# 6. Zmiana skladki wybranych kierowcow w czasie
selected = latest_premium.sort_values('technical_premium', ascending=False)['driver_id'].head(5).tolist()
hist = premium[premium['driver_id'].isin(selected)].copy()
hist['scored_at'] = pd.to_datetime(hist['scored_at'])
for driver_id, group in hist.groupby('driver_id'):
    plt.plot(group['scored_at'], group['technical_premium'], marker='o', label=driver_id)
plt.title('Zmiana skladki wybranych kierowcow')
plt.ylabel('PLN')
plt.legend()
plt.xticks(rotation=30)
plt.show()

In [ ]:
# 7. Porownanie profili safe / average / aggressive
if 'driver_profile' in features.columns:
    profile_summary = features.groupby('driver_profile')[['speeding_ratio', 'night_event_ratio', 'bad_weather_event_ratio', 'risk_score_driver']].mean()
    profile_summary.loc[[p for p in ['safe', 'average', 'aggressive'] if p in profile_summary.index]].plot(
        kind='bar', figsize=(10, 4), title='Porownanie profili kierowcow'
    )
    plt.ylabel('Srednia wartosc')
    plt.show()

## 8. Krytyczna ocena

Dane sa syntetyczne, wiec nie mozna wyciagac realnych wnioskow taryfowych. Prawdziwa skladka zalezy od wielu innych zmiennych niz telematyka, np. historii polisowej, regionu, pojazdu, wieku kierowcy i zakresu ochrony. Pojedyncze zdarzenia nie powinny mechanicznie zmieniac skladki, bo moga byc przypadkowe lub wynikac z kontekstu drogowego.

GLM jest dobry dydaktycznie, bo jest interpretowalny, ale moze byc zbyt prosty dla zlozonych zaleznosci telematycznych. Poisson zaklada okreslona relacje sredniej i wariancji; przy nadmiernej dyspersji lepszy moze byc Negative Binomial albo Tweedie. Telematyka wymaga tez rozwazenia prywatnosci, zgody klienta, celu przetwarzania i retencji danych.

Kafka i Spark sa uzasadnione, gdy dane sa strumieniowe lub duze. W malej probce projekt jest przede wszystkim demonstracja architektury real-time analytics.